# Portfolio-safe version

This notebook is a sanitized portfolio adaptation of the author's MSc Data Analytics project.
Environment-specific paths, cloud bucket names, notebook outputs, and exact patient examples
have been removed or generalized. The original methodology and core code structure are preserved.

**Data note:** the underlying TCIA imaging/clinical data are not redistributed in this repository.
Configure your own authorized/local dataset paths before running the notebook.


In [ ]:
# Install once if needed: pip install dcmrtstruct2nii
# Install once if needed: pip install pydicom
# Install once if needed: pip install nibabel
# Install once if needed: pip install google-cloud-storage

In [ ]:
import os
import json
import shutil
import numpy as np
import pydicom
import nibabel as nib
from tqdm import tqdm
import psutil
import radiomics
from radiomics import featureextractor
from google.cloud import storage
import logging
import time

# Suppress excessive PyRadiomics warnings (e.g., GLCM symmetry)
logging.getLogger('radiomics.glcm').setLevel(logging.ERROR)


# =====================
# CONFIG
# =====================
GCS_BUCKET = os.getenv("GCS_BUCKET", "your-gcs-bucket")
PREFIX_SRC = os.getenv("GCS_SOURCE_URI", f"gs://{GCS_BUCKET}/path/to/soft-tissue-sarcoma")
PREFIX_DST = os.getenv("GCS_OUTPUT_URI", f"gs://{GCS_BUCKET}/radiomics-output")
WORKDIR = "data/workdir"
os.makedirs(WORKDIR, exist_ok=True)

# Load mapping CSV
import pandas as pd
mapping_df = pd.read_csv("data/rtstruct_dicom_mapping.csv")
total_cases = len(mapping_df)
print(f"Total mapped cases: {total_cases}")
print("Starting full radiomics extraction for all mapped cases...")

# GCS client
gcs_client = storage.Client()

# PyRadiomics extractor with label=255
params = {
    'binWidth': 25,
    'resampledPixelSpacing': None,
    'interpolator': 'sitkBSpline',
    'label': 255
}
extractor = featureextractor.RadiomicsFeatureExtractor(**params)


# =====================
# Helper: JSON-safe conversion
# =====================
def make_json_serializable(obj):
    """Recursively convert NumPy objects into JSON-safe types."""
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(x) for x in obj]
    else:
        return obj


# =====================
# GCS download helper (with retry)
# =====================
def download_gcs_folder(bucket_name, prefix, local_path, retries=3):
    """Download all files from a GCS prefix into a local folder, with retry on failure."""
    for attempt in range(retries):
        try:
            bucket = gcs_client.bucket(bucket_name)
            blobs = list(bucket.list_blobs(prefix=prefix))
            if not blobs:
                print(f"  No files found in GCS prefix: {prefix}")
                return False
            os.makedirs(local_path, exist_ok=True)
            for blob in blobs:
                if blob.name.endswith('/'):
                    continue
                dest_path = os.path.join(local_path, os.path.basename(blob.name))
                blob.download_to_filename(dest_path)
            return True
        except Exception as e:
            print(f"  WARNING: GCS download failed (attempt {attempt+1}/{retries}): {e}")
            time.sleep(3)
    return False


# =====================
# Load DICOM as 3D volume
# =====================
def load_dicom_series(path):
    slices = [
        pydicom.dcmread(os.path.join(path, f))
        for f in os.listdir(path) if f.endswith(".dcm")
    ]
    slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))  # sort by Z position

    volume = np.stack([s.pixel_array for s in slices], axis=-1)

    spacing = list(slices[0].PixelSpacing) + [float(slices[0].SliceThickness)]
    affine = np.diag(spacing + [1])  # affine for NIfTI

    return volume, affine, slices


# =====================
# Parse RTSTRUCT → mask volume
# =====================
def parse_rtstruct_to_mask(rtstruct_path, dicom_slices):
    rtstruct = pydicom.dcmread(rtstruct_path)
    mask = np.zeros((dicom_slices[0].Rows, dicom_slices[0].Columns, len(dicom_slices)), dtype=np.uint8)

    # Z → slice index mapping
    z_map = {round(float(s.ImagePositionPatient[2]), 3): idx for idx, s in enumerate(dicom_slices)}
    print(f"  DICOM slices: {len(z_map)} | sample Z positions: {list(z_map.keys())[:5]} ...")

    if not hasattr(rtstruct, "ROIContourSequence") or not hasattr(rtstruct, "StructureSetROISequence"):
        print("  RTSTRUCT has no ROIContourSequence, skipping.")
        return mask

    # Map ROI number → ROI name
    roi_names = {}
    for roi in rtstruct.StructureSetROISequence:
        roi_num = roi.ROINumber
        roi_name = roi.ROIName if hasattr(roi, "ROIName") else f"ROI_{roi_num}"
        roi_names[roi_num] = roi_name

    print(f"  RTSTRUCT contains {len(rtstruct.ROIContourSequence)} ROIs")

    def find_nearest_slice(z_val):
        return min(z_map.keys(), key=lambda k: abs(k - z_val))

    import cv2

    for roi_idx, roi_contour in enumerate(rtstruct.ROIContourSequence):
        roi_number = roi_contour.ReferencedROINumber
        roi_name = roi_names.get(roi_number, f"ROI_{roi_number}")
        contour_count = len(roi_contour.ContourSequence)

        print(f"    ROI[{roi_idx}] '{roi_name}' has {contour_count} contours")

        for contour in roi_contour.ContourSequence:
            coords = np.array(contour.ContourData).reshape(-1, 3)
            z = float(coords[0, 2])

            # Match contour Z to nearest slice Z
            nearest_z = find_nearest_slice(z)
            slice_idx = z_map[nearest_z]

            # Convert physical coords → pixel indices
            x = coords[:, 0] / dicom_slices[0].PixelSpacing[0]
            y = coords[:, 1] / dicom_slices[0].PixelSpacing[1]
            poly = np.vstack((x, y)).T.astype(np.int32)

            img = np.zeros((dicom_slices[0].Rows, dicom_slices[0].Columns), dtype=np.uint8)
            cv2.fillPoly(img, [poly], 255)
            mask[:, :, slice_idx] = np.maximum(mask[:, :, slice_idx], img)

    return mask


# =====================
# Process single case
# =====================
def process_case(row, idx):
    case_id = row['case_id']
    study = row['study_info']
    dicom_series = row['matched_dicom_series']
    rtstruct_series = row['rtstruct_series']

    print("=" * 60)
    print(f"Processing case {idx+1}/{total_cases}")
    print(f"Case ID: {case_id}")
    print(f"Study: {study}")
    print(f"DICOM series: {dicom_series}")
    print(f"RTSTRUCT: {rtstruct_series}")
    print("=" * 60)

    # ✅ FIX: use dynamic case_id instead of hardcoding STS_XXX
    dicom_prefix = f"{case_id}/{study}/{dicom_series}"
    rtstruct_prefix = f"{case_id}/{study}/{rtstruct_series}"

    local_dicom = os.path.join(WORKDIR, f"case_{idx}_dicom")
    local_rtstruct = os.path.join(WORKDIR, f"case_{idx}_rtstruct")

    # Download DICOM + RTSTRUCT
    if not download_gcs_folder('your-gcs-bucket', f"path/to/soft-tissue-sarcoma/{dicom_prefix}", local_dicom):
        print(f"  ERROR: Failed to download DICOM for {case_id}")
        return {'case_id': case_id, 'status': 'failed', 'error': 'dicom_download_failed'}

    if not download_gcs_folder('your-gcs-bucket', f"path/to/soft-tissue-sarcoma/{rtstruct_prefix}", local_rtstruct):
        print(f"  ERROR: Failed to download RTSTRUCT for {case_id}")
        return {'case_id': case_id, 'status': 'failed', 'error': 'rtstruct_download_failed'}

    rtstruct_files = [os.path.join(local_rtstruct, f) for f in os.listdir(local_rtstruct) if f.endswith('.dcm')]
    if not rtstruct_files:
        print(f"  ERROR: No RTSTRUCT found for {case_id}")
        return {'case_id': case_id, 'status': 'failed', 'error': 'rtstruct_missing'}
    rtstruct_file = rtstruct_files[0]

    # Load DICOM → volume
    volume, affine, slices = load_dicom_series(local_dicom)

    # RTSTRUCT → mask
    mask = parse_rtstruct_to_mask(rtstruct_file, slices)
    unique_vals = np.unique(mask)

    if 255 not in unique_vals:
        print(f"  WARNING: Mask for {case_id} has no ROI (unique={unique_vals}), skipping radiomics.")
        shutil.rmtree(local_dicom)
        shutil.rmtree(local_rtstruct)
        return {'case_id': case_id, 'status': 'skipped', 'error': 'mask_empty'}

    # Save NIfTI
    output_dir = os.path.join(WORKDIR, f"case_{idx}_output")
    os.makedirs(output_dir, exist_ok=True)
    image_nii = os.path.join(output_dir, "image.nii.gz")
    mask_nii = os.path.join(output_dir, "mask.nii.gz")
    nib.save(nib.Nifti1Image(volume, affine), image_nii)
    nib.save(nib.Nifti1Image(mask, affine), mask_nii)

    # Radiomics extraction
    print("  Extracting radiomics features...")
    features = extractor.execute(image_nii, mask_nii)

    # Convert to JSON-safe
    features_safe = make_json_serializable(features)

    # Save JSON per case
    features_json_path = os.path.join(output_dir, "features.json")
    with open(features_json_path, 'w') as f:
        json.dump(features_safe, f, indent=2)

    # Upload outputs to GCS
    dst_prefix = f"STS_Pyradiomics/{case_id}/{study}/{dicom_series}"
    bucket = gcs_client.bucket(GCS_BUCKET)
    for fpath in [image_nii, mask_nii, features_json_path]:
        blob = bucket.blob(f"{dst_prefix}/{os.path.basename(fpath)}")
        blob.upload_from_filename(fpath)

    # Cleanup local temp dirs
    shutil.rmtree(local_dicom)
    shutil.rmtree(local_rtstruct)
    shutil.rmtree(output_dir)

    # Log resource usage
    cpu_usage = psutil.cpu_percent()
    mem_usage = psutil.virtual_memory().percent
    print(f"  Completed {case_id} | CPU: {cpu_usage}% | RAM: {mem_usage}%")

    return {'case_id': case_id, 'status': 'success', 'cpu': cpu_usage, 'ram': mem_usage}


# =====================
# Run all cases
# =====================
report = []
for idx in tqdm(range(total_cases)):
    row = mapping_df.iloc[idx]
    result = process_case(row, idx)
    report.append(result)

# Save global summary
summary_path = os.path.join(WORKDIR, "summary_report.json")
with open(summary_path, 'w') as f:
    json.dump(report, f, indent=2)

print("===== FULL RUN COMPLETE =====")
print(f"Summary saved to: {summary_path}")


In [ ]:
import os
import json
import shutil
import numpy as np
import pydicom
import nibabel as nib
from tqdm import tqdm
import psutil
import radiomics
from radiomics import featureextractor
from google.cloud import storage
import logging
import time
import pandas as pd

# Suprimir warnings desnecessários do PyRadiomics
logging.getLogger('radiomics.glcm').setLevel(logging.ERROR)


# =====================
# CONFIG
# =====================
GCS_BUCKET = os.getenv("GCS_BUCKET", "your-gcs-bucket")
PREFIX_SRC = os.getenv("GCS_SOURCE_URI", f"gs://{GCS_BUCKET}/path/to/soft-tissue-sarcoma")
PREFIX_DST = os.getenv("GCS_OUTPUT_URI", f"gs://{GCS_BUCKET}/radiomics-output")
WORKDIR = "data/workdir"
os.makedirs(WORKDIR, exist_ok=True)

# Carregar mapeamento
mapping_df = pd.read_csv("data/rtstruct_dicom_mapping.csv")

# ✅ Defina o ponto de retomada
start_case = "STS_XXX"  # <-- altere aqui se quiser outro ponto de retomada

if start_case in mapping_df['case_id'].values:
    start_idx = mapping_df.index[mapping_df['case_id'] == start_case].tolist()[0]
    mapping_df = mapping_df.iloc[start_idx:].reset_index(drop=True)
    print(f"Retomando a partir do caso {start_case} (índice {start_idx}), total restante: {len(mapping_df)} casos")
else:
    print(f"ATENÇÃO: {start_case} não encontrado no mapeamento. Processando todos os casos!")

total_cases = len(mapping_df)

# Cliente GCS
gcs_client = storage.Client()

# Config do PyRadiomics
params = {
    'binWidth': 25,
    'resampledPixelSpacing': None,
    'interpolator': 'sitkBSpline',
    'label': 255
}
extractor = featureextractor.RadiomicsFeatureExtractor(**params)

# =====================
# Função JSON-safe
# =====================
def make_json_serializable(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    elif isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    elif isinstance(obj, dict):
        return {k: make_json_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [make_json_serializable(x) for x in obj]
    else:
        return obj

# =====================
# Função download GCS
# =====================
def download_gcs_folder(bucket_name, prefix, local_path, retries=3):
    for attempt in range(retries):
        try:
            bucket = gcs_client.bucket(bucket_name)
            blobs = list(bucket.list_blobs(prefix=prefix))
            if not blobs:
                print(f"  Nenhum arquivo encontrado em {prefix}")
                return False
            os.makedirs(local_path, exist_ok=True)
            for blob in blobs:
                if blob.name.endswith('/'):
                    continue
                dest_path = os.path.join(local_path, os.path.basename(blob.name))
                blob.download_to_filename(dest_path)
            return True
        except Exception as e:
            print(f"  WARNING: Falha no download (tentativa {attempt+1}/{retries}): {e}")
            time.sleep(3)
    return False

# =====================
# Carregar DICOM em volume 3D
# =====================
def load_dicom_series(path):
    # Carrega todas as slices DICOM
    slices = [
        pydicom.dcmread(os.path.join(path, f))
        for f in os.listdir(path) if f.endswith(".dcm")
    ]
    slices.sort(key=lambda s: float(s.ImagePositionPatient[2]))

    # Constrói volume 3D
    volume = np.stack([s.pixel_array for s in slices], axis=-1)

    # ✅ Pixel Spacing sempre deveria existir, mas tratamos fallback
    if hasattr(slices[0], "PixelSpacing"):
        px_spacing = [float(v) for v in slices[0].PixelSpacing]
    else:
        px_spacing = [1.0, 1.0]  # fallback
        print("  [WARNING] PixelSpacing ausente → usando 1.0 mm x 1.0 mm")

    # ✅ SliceThickness pode estar ausente
    slice_thickness = getattr(slices[0], "SliceThickness", None)

    if slice_thickness is None:
        # ✅ Inferir pela diferença entre Z das fatias consecutivas
        if len(slices) > 1:
            try:
                z_positions = [float(s.ImagePositionPatient[2]) for s in slices]
                inferred_spacing = abs(z_positions[1] - z_positions[0])
                slice_thickness = inferred_spacing
                print(f"  [WARNING] SliceThickness ausente → inferido {slice_thickness:.3f} mm")
            except Exception:
                slice_thickness = 1.0
                print("  [WARNING] Falha ao inferir SliceThickness → usando 1.0 mm")
        else:
            slice_thickness = 1.0
            print("  [WARNING] Apenas 1 slice encontrado → usando SliceThickness = 1.0 mm")

    else:
        try:
            slice_thickness = float(slice_thickness)
        except Exception:
            slice_thickness = 1.0
            print("  [WARNING] SliceThickness inválido → usando 1.0 mm")

    # ✅ Monta spacing final e affine
    spacing = px_spacing + [slice_thickness]
    affine = np.diag(spacing + [1])  # affine para NIfTI

    return volume, affine, slices


# =====================
# Converter RTSTRUCT → máscara
# =====================
def parse_rtstruct_to_mask(rtstruct_path, dicom_slices):
    rtstruct = pydicom.dcmread(rtstruct_path)
    mask = np.zeros((dicom_slices[0].Rows, dicom_slices[0].Columns, len(dicom_slices)), dtype=np.uint8)
    z_map = {round(float(s.ImagePositionPatient[2]), 3): idx for idx, s in enumerate(dicom_slices)}

    if not hasattr(rtstruct, "ROIContourSequence") or not hasattr(rtstruct, "StructureSetROISequence"):
        print("  RTSTRUCT sem ROIContourSequence, ignorando.")
        return mask

    roi_names = {roi.ROINumber: getattr(roi, "ROIName", f"ROI_{roi.ROINumber}") for roi in rtstruct.StructureSetROISequence}
    import cv2

    def find_nearest_slice(z_val):
        return min(z_map.keys(), key=lambda k: abs(k - z_val))

    for roi_contour in rtstruct.ROIContourSequence:
        roi_number = roi_contour.ReferencedROINumber
        roi_name = roi_names.get(roi_number, f"ROI_{roi_number}")
        print(f"  ROI '{roi_name}' com {len(roi_contour.ContourSequence)} contornos")

        for contour in roi_contour.ContourSequence:
            coords = np.array(contour.ContourData).reshape(-1, 3)
            z = float(coords[0, 2])
            nearest_z = find_nearest_slice(z)
            slice_idx = z_map[nearest_z]

            x = coords[:, 0] / dicom_slices[0].PixelSpacing[0]
            y = coords[:, 1] / dicom_slices[0].PixelSpacing[1]
            poly = np.vstack((x, y)).T.astype(np.int32)

            img = np.zeros((dicom_slices[0].Rows, dicom_slices[0].Columns), dtype=np.uint8)
            cv2.fillPoly(img, [poly], 255)
            mask[:, :, slice_idx] = np.maximum(mask[:, :, slice_idx], img)

    return mask

# =====================
# Processar um caso
# =====================
def process_case(row, idx):
    case_id = row['case_id']
    study = row['study_info']
    dicom_series = row['matched_dicom_series']
    rtstruct_series = row['rtstruct_series']

    print("="*60)
    print(f"Processando caso {idx+1}/{total_cases} → {case_id}")
    print("="*60)

    dicom_prefix = f"{case_id}/{study}/{dicom_series}"
    rtstruct_prefix = f"{case_id}/{study}/{rtstruct_series}"

    local_dicom = os.path.join(WORKDIR, f"case_{idx}_dicom")
    local_rtstruct = os.path.join(WORKDIR, f"case_{idx}_rtstruct")

    if not download_gcs_folder('your-gcs-bucket', f"path/to/soft-tissue-sarcoma/{dicom_prefix}", local_dicom):
        return {'case_id': case_id, 'status': 'failed', 'error': 'dicom_download_failed'}

    if not download_gcs_folder('your-gcs-bucket', f"path/to/soft-tissue-sarcoma/{rtstruct_prefix}", local_rtstruct):
        return {'case_id': case_id, 'status': 'failed', 'error': 'rtstruct_download_failed'}

    rtstruct_files = [os.path.join(local_rtstruct, f) for f in os.listdir(local_rtstruct) if f.endswith('.dcm')]
    if not rtstruct_files:
        return {'case_id': case_id, 'status': 'failed', 'error': 'rtstruct_missing'}
    rtstruct_file = rtstruct_files[0]

    volume, affine, slices = load_dicom_series(local_dicom)
    mask = parse_rtstruct_to_mask(rtstruct_file, slices)

    if 255 not in np.unique(mask):
        print(f"⚠️ Máscara vazia para {case_id}, ignorando radiômica.")
        shutil.rmtree(local_dicom)
        shutil.rmtree(local_rtstruct)
        return {'case_id': case_id, 'status': 'skipped', 'error': 'mask_empty'}

    output_dir = os.path.join(WORKDIR, f"case_{idx}_output")
    os.makedirs(output_dir, exist_ok=True)
    image_nii = os.path.join(output_dir, "image.nii.gz")
    mask_nii = os.path.join(output_dir, "mask.nii.gz")
    nib.save(nib.Nifti1Image(volume, affine), image_nii)
    nib.save(nib.Nifti1Image(mask, affine), mask_nii)

    print("Extraindo features radiômicas...")
    features = extractor.execute(image_nii, mask_nii)
    features_safe = make_json_serializable(features)

    features_json_path = os.path.join(output_dir, "features.json")
    with open(features_json_path, 'w') as f:
        json.dump(features_safe, f, indent=2)

    dst_prefix = f"STS_Pyradiomics/{case_id}/{study}/{dicom_series}"
    bucket = gcs_client.bucket(GCS_BUCKET)
    for fpath in [image_nii, mask_nii, features_json_path]:
        blob = bucket.blob(f"{dst_prefix}/{os.path.basename(fpath)}")
        blob.upload_from_filename(fpath)

    shutil.rmtree(local_dicom)
    shutil.rmtree(local_rtstruct)
    shutil.rmtree(output_dir)

    cpu_usage = psutil.cpu_percent()
    mem_usage = psutil.virtual_memory().percent
    print(f"✅ Concluído {case_id} | CPU: {cpu_usage}% | RAM: {mem_usage}%")

    return {'case_id': case_id, 'status': 'success', 'cpu': cpu_usage, 'ram': mem_usage}

# =====================
# Loop de execução
# =====================
report = []
for idx in tqdm(range(total_cases)):
    row = mapping_df.iloc[idx]
    result = process_case(row, idx)
    report.append(result)

summary_path = os.path.join(WORKDIR, "summary_report.json")
with open(summary_path, 'w') as f:
    json.dump(report, f, indent=2)

print("===== PROCESSAMENTO COMPLETO =====")
print(f"Resumo salvo em: {summary_path}")
